In [2]:
import pandas as pd
import subprocess

In [38]:
distilled_proteins_dir = "toydata1"
rfdiffusion_proteins_dir = "toydata2"
temp_dir = "tmp"

Calculate Precision and Recall

In [57]:
precision_output = "precision_output.m8"
recall_output = "recall_output.m8"

# Example call: foldseek easy-search toydata2/ toydata2/1GYX.pdb.gz result.m8 tmp --format-output "query,alntmscore" --alignment-type 1
# We only save the name of the protein and its alignment score because we don't care which target it was closest to.
subprocess.run(["foldseek", "easy-search", distilled_proteins_dir, rfdiffusion_proteins_dir, precision_output, temp_dir, '--format-output', "query,alntmscore", "-v", "0", "--alignment-type", "1"])
subprocess.run(["foldseek", "easy-search", rfdiffusion_proteins_dir, distilled_proteins_dir, recall_output, temp_dir, '--format-output', "query,alntmscore", "-v", "0", "--alignment-type", "1"])

# Import the results files and count how many proteins have a maximum TM score greater than 0.5
precision_df =  pd.read_csv(precision_output, sep=r"\s+", names = ["Query", "TM"])
recall_df    =  pd.read_csv(recall_output,    sep=r"\s+", names = ["Query", "TM"])

def prop_above_tm_cutoff(df, cutoff):
    d = (df.groupby("Query").max() > cutoff)["TM"]
    return sum(d) / len(d)

precision = prop_above_tm_cutoff(precision_df, 0.5)
recall    = prop_above_tm_cutoff(recall_df,    0.5)

print(f"Precision: {precision}, Recall: {recall}")

Diversity

In [55]:
diversity_output = "diversity"

#Example run: foldseek easy-cluster toydata1 result tmp --alignment-type 1 --tmscore-threshold 0.6
# We use a TM score of 0.6 because that is what was used in the RFDiffusion paper.
tmscore_threshold = str(0.6)
subprocess.run(["foldseek", "easy-cluster", distilled_proteins_dir, diversity_output, temp_dir, "--alignment-type", "1", "--tmscore-threshold", tmscore_threshold, "-v", "0"])

# Import the result file and count how many clusters there are and then divide by the number of proteins to get diversity
diversity_df = pd.read_csv(f"{diversity_output}_cluster.tsv", names = ["Representative", "Member"], sep="\t")
d = diversity_df["Representative"]
diversity = len(d.unique()) / len(d)
print(f"Diversity: {diversity}")

Diversity: 0.3048780487804878
